<a href="https://colab.research.google.com/github/sohammondal29/QML-Supply-Chain/blob/main/QML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q lightgbm xgboost catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# Load datasets
train_df = pd.read_csv("Training_Top3Features.csv")
test_df = pd.read_csv("Testing_Top3Features.csv")

X_train = train_df.drop(columns=['went_on_backorder'])
y_train = train_df['went_on_backorder']

X_test = test_df.drop(columns=['went_on_backorder'])
y_test = test_df['went_on_backorder']

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


# Define models and hyperparameters
models = {

    "CatBoost": (
        CatBoostClassifier(
            silent=True,
            random_state=42
        ),
        {
            'depth': [4, 6],
            'learning_rate': [0.01, 0.1],
            'iterations': [50, 100]
        }
    ),

    "LGBM": (
        LGBMClassifier(
            random_state=42,
            verbosity=-1
        ),
        {
            'num_leaves': [15, 31],
            'learning_rate': [0.01, 0.1],
            'n_estimators': [50, 100]
        }
    ),

    "RandomForest": (
        RandomForestClassifier(
            random_state=42
        ),
        {
            'n_estimators': [50, 100],
            'max_depth': [5, None]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            eval_metric='logloss',
            random_state=42
        ),
        {
            'max_depth': [3, 6],
            'learning_rate': [0.01, 0.1],
            'n_estimators': [50, 100]
        }
    ),

    "ANN": (
        MLPClassifier(
            random_state=42,
            max_iter=500
        ),
        {
            'hidden_layer_sizes': [(14, 14, 10)],
            'activation': ['relu'],
            'learning_rate_init': [0.001, 0.01]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            'n_neighbors': [3, 5, 7]
        }
    ),

    "SVM": (
        SVC(
            probability=True,
            random_state=42
        ),
        {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf']
        }
    ),

    "DecisionTree": (
        DecisionTreeClassifier(
            random_state=42
        ),
        {
            'max_depth': [5, None],
            'criterion': ['gini', 'entropy']
        }
    )
}


print("\nModels successfully defined:")
print(list(models.keys()))

Training shape: (10000, 3)
Testing shape: (4000, 3)

Models successfully defined:
['CatBoost', 'LGBM', 'RandomForest', 'XGBoost', 'ANN', 'KNN', 'SVM', 'DecisionTree']


In [6]:
results = []

for name, (model, params) in models.items():

    print(f"\n🔷 Training {name} ...")

    clf = GridSearchCV(
        estimator=model,
        param_grid=params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring='f1'
    )

    clf.fit(X_train, y_train)

    best_model = clf.best_estimator_

    # Predictions
    preds = best_model.predict(X_test)

    # Probabilities
    if hasattr(best_model, "predict_proba"):
        probs = best_model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, probs)
    else:
        roc_auc = "N/A"

    # Metrics
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    print(f"✅ {name} Best Params: {clf.best_params_}")
    print(f"✅ {name} Test Accuracy: {acc:.4f}")
    print(f"✅ {name} F1 Score: {f1:.4f}")
    print(f"✅ {name} ROC AUC: {roc_auc}")
    print(f"✅ {name} Confusion Matrix:\n{cm}")

    results.append({
        "Model": name,
        "Best Params": clf.best_params_,
        "Accuracy": acc,
        "F1": f1,
        "ROC AUC": roc_auc,
        "Confusion Matrix": cm.tolist()
    })


🔷 Training CatBoost ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
✅ CatBoost Best Params: {'depth': 6, 'iterations': 100, 'learning_rate': 0.1}
✅ CatBoost Test Accuracy: 0.8117
✅ CatBoost F1 Score: 0.8046
✅ CatBoost ROC AUC: 0.8740206249999999
✅ CatBoost Confusion Matrix:
[[1697  303]
 [ 450 1550]]

🔷 Training LGBM ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
✅ LGBM Best Params: {'learning_rate': 0.1, 'n_estimators': 50, 'num_leaves': 15}
✅ LGBM Test Accuracy: 0.8125
✅ LGBM F1 Score: 0.8080
✅ LGBM ROC AUC: 0.8740001249999999
✅ LGBM Confusion Matrix:
[[1672  328]
 [ 422 1578]]

🔷 Training RandomForest ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
✅ RandomForest Best Params: {'max_depth': 5, 'n_estimators': 100}
✅ RandomForest Test Accuracy: 0.8067
✅ RandomForest F1 Score: 0.8035
✅ RandomForest ROC AUC: 0.8658359999999999
✅ RandomForest Confusion Matrix:
[[1647  353]
 [ 420 1580]]

🔷 Training XGBoost ...
Fitting 3 folds for each of 